# Project: A global neural forecaster ⚡🌍
Forecast tomorrow's 24 hourly electricity prices in **six price areas** (DK1, DK2, SE3, SE4, NO2, DE) from the last week, and test the main claims of [[Deep Learning for Time Series]] on real data:
1. **One global model** trained on all areas beats **one model per area**.
2. **Instance normalisation (RevIN)** lets one model handle areas with very different price levels.
3. A **temporal convolutional network** with **quantile outputs** gives prediction intervals, and you must check whether they're calibrated.
4. And the DLinear lesson: a well-built **linear** model is a baseline neural nets have to *earn* their way past.

Topic: [[Deep Learning for Time Series]], [[Time Series Forecasting]] · exercises: [Deep Learning for Time Series - Exercises](Deep%20Learning%20for%20Time%20Series%20-%20Exercises.ipynb) · guide note: [[Project - Global Neural Forecaster]] · data: [Energi Data Service](https://www.energidataservice.dk/), as in [[Danish Electricity Prices]]

Runtime ≈ 1 minute on a CPU.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, time, urllib.parse, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
AREAS = ["DK1", "DK2", "SE3", "SE4", "NO2", "DE"]
f = pathlib.Path("data/Elspotprices_multi_2023-10_2025-10.csv"); f.parent.mkdir(exist_ok=True)
if not f.exists():
    q = {"start": "2023-10-01", "end": "2025-10-01", "filter": json.dumps({"PriceArea": AREAS}), "limit": 0}
    pd.DataFrame(json.load(urllib.request.urlopen("https://api.energidataservice.dk/dataset/Elspotprices?" + urllib.parse.urlencode(q)))["records"]).to_csv(f, index=False)
d = pd.read_csv(f); P = d.pivot_table(index="HourUTC", columns="PriceArea", values="SpotPriceEUR").sort_index()
P.index = pd.to_datetime(P.index); P = P.asfreq("h").interpolate(limit=3)
print(P.describe().loc[["mean", "std"]].round(1))
P[AREAS].resample("W").mean().plot(figsize=(12, 3), title="weekly mean price (EUR/MWh): same shape, different levels"); plt.show()

L, H = 168, 24                                          # look back one week, forecast the next day
def windows(series, start, end):
    """One window per day: origin at 22:00 UTC, input = previous 168 h, target = next 24 h."""
    v = series.to_numpy(); X, Y = [], []
    for i in np.where((series.index >= start) & (series.index < end) & (series.index.hour == 22))[0]:
        if i - L >= 0 and i + H <= len(v): X.append(v[i - L:i]); Y.append(v[i:i + H])
    return np.array(X), np.array(Y)
train = {a: windows(P[a], "2023-10-08", "2025-01-01") for a in AREAS}; test = {a: windows(P[a], "2025-01-01", "2025-10-01") for a in AREAS}
X_tr = np.concatenate([train[a][0] for a in AREAS]); Y_tr = np.concatenate([train[a][1] for a in AREAS])
X_te = np.concatenate([test[a][0] for a in AREAS]); Y_te = np.concatenate([test[a][1] for a in AREAS])
mae = lambda a, b_: float(np.mean(np.abs(a - b_)))
results = {"seasonal naive (yesterday)": mae(X_te[:, -24:], Y_te), "seasonal naive (last week)": mae(X_te[:, :24], Y_te)}
print(f"{len(train['DK1'][0])} training days per area × 6 areas = {len(X_tr)} windows; {len(X_te)} test windows (2025)")
for k, v in results.items(): print(f"{k:<28} MAE {v:.2f}")

## 1. RevIN: normalise every window by itself
Prices in SE3 average ~41 EUR/MWh, in DE ~83. Subtract each window's own mean and divide by its own standard deviation (+ `eps`), so the model sees *shapes*, not levels. Return `(X_normalised, (mu, sd))` with `mu, sd` of shape `(n, 1)`; to de-normalise a forecast: `y_norm * sd + mu`.

In [ ]:
def revin(X, eps=1e-3):
    # TODO 1: per-row mean and std (keepdims), return normalised X and the stats
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_revin():
    Xn, (mu, sd) = revin(np.array([[10.0, 12, 14, 12], [100, 100, 100, 100]]))
    return np.allclose(Xn[0], np.array([-1, 0, 1, 0]) * np.sqrt(2), atol=1e-3) and np.allclose(Xn[1], 0) and mu.shape == (2, 1) and np.allclose(Xn * sd + mu, [[10, 12, 14, 12], [100] * 4])
check("RevIN", _t_revin, "[10,12,14,12] → [−1.41, 0, 1.41, 0]; a constant window → 0 (thanks to eps); de-normalising must give the input back.")

## 2. Global vs. local linear models
A linear map from the 168 normalised inputs to the 24 outputs (+ bias), fitted by least squares. **Local:** one per area (451 training days for 169 × 24 weights). **Global:** one for all six areas pooled. Same model class, different data.

In [ ]:
def linear_forecaster(X, Y):
    Xn, (mu, sd) = revin(X); A = np.c_[Xn, np.ones(len(X))]
    W = np.linalg.lstsq(A, (Y - mu) / sd, rcond=None)[0]
    def predict(Xt):
        Xtn, (mt, st) = revin(Xt); return (np.c_[Xtn, np.ones(len(Xt))] @ W) * st + mt
    return predict
if ready("RevIN"):
    local_err = np.concatenate([np.abs(linear_forecaster(*train[a])(test[a][0]) - test[a][1]).ravel() for a in AREAS])
    results["local linear (one per area)"] = float(local_err.mean())
    glob = linear_forecaster(X_tr, Y_tr); results["global linear + RevIN"] = mae(glob(X_te), Y_te)
    for k in ("local linear (one per area)", "global linear + RevIN"): print(f"{k:<28} MAE {results[k]:.2f}")
    check("global beats local", lambda: results["global linear + RevIN"] < 0.85 * results["local linear (one per area)"] and results["global linear + RevIN"] < results["seasonal naive (yesterday)"],
          "The pooled model should beat per-area models by > 15 % and beat seasonal naive.")

## 3. A causal convolution layer
The building block of a TCN: a 1-D convolution that only looks at the present and the past. Pad the input with `(kernel_size − 1) · dilation` zeros **on the left only** (`F.pad(x, (pad, 0))`) and apply `self.conv` (which has no padding of its own). Output length = input length.

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, c_in, c_out, kernel_size, dilation):
        super().__init__(); self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel_size, dilation=dilation)
    def forward(self, x):                 # x: (batch, channels, time)
        # TODO 2: left-pad, then convolve
        raise NotImplementedError  # TODO 2

In [ ]:
def _t_causal():
    torch.manual_seed(0); layer = CausalConv1d(1, 2, 3, dilation=4); x = torch.randn(1, 1, 50)
    y1 = layer(x); x2 = x.clone(); x2[..., 30:] += 5; y2 = layer(x2)
    return y1.shape == (1, 2, 50) and torch.allclose(y1[..., :30], y2[..., :30]) and not torch.allclose(y1[..., 30:], y2[..., 30:])
check("causal conv", _t_causal, "Same length as the input; outputs before t=30 must not change when inputs from t=30 on change.")

## 4. Quantile (pinball) loss
The network outputs three curves: the 10th, 50th and 90th percentile. For each quantile $q$ and error $u = y - \hat y_q$: $\rho_q(u)=\max(qu, (q-1)u)$; average over batch and horizon and **sum over the quantiles**. `pred` has shape `(batch, 3, 24)`, `y` `(batch, 24)`.

In [ ]:
def pinball_loss(pred, y, quantiles=(0.1, 0.5, 0.9)):
    # TODO 3: summed mean pinball losses of the quantile outputs
    raise NotImplementedError  # TODO 3

In [ ]:
def _t_pin():
    y = torch.tensor([[50.0]]); pred = torch.tensor([[[40.0], [50.0], [60.0]]])
    return abs(pinball_loss(pred, y).item() - (1.0 + 0.0 + 1.0)) < 1e-6 and abs(pinball_loss(torch.tensor([[[60.0], [50.0], [40.0]]]), y).item() - (9.0 + 0 + 9.0)) < 1e-6
check("pinball loss", _t_pin, "Truth 50, quantile forecasts (40, 50, 60) → 1 + 0 + 1; reversed (60, 50, 40) → 9 + 0 + 9.")

## 5. Train a global TCN with quantile outputs
Six causal dilated layers (receptive field 127 h) with residual connections; the last 24 time steps of the feature map go through a linear head to 3 × 24 outputs. Trained on RevIN-normalised windows from all areas.

In [ ]:
class TCN(nn.Module):
    def __init__(self, ch=32, k=3, dilations=(1, 2, 4, 8, 16, 32)):
        super().__init__()
        self.layers = nn.ModuleList([CausalConv1d(1 if i == 0 else ch, ch, k, d) for i, d in enumerate(dilations)])
        self.head = nn.Linear(ch * 24, 3 * H)
    def forward(self, x):
        h = x.unsqueeze(1)
        for i, layer in enumerate(self.layers):
            out = F.relu(layer(h)); h = out if i == 0 else h + out            # residual connections
        return self.head(h[:, :, -24:].flatten(1)).view(-1, 3, H)

if ready("RevIN", "causal conv", "pinball loss"):
    Xn, (mu, sd) = revin(X_tr); Yn = (Y_tr - mu) / sd; Xtn, (mt, st) = revin(X_te)
    Xn, Yn = torch.tensor(Xn, dtype=torch.float32), torch.tensor(Yn, dtype=torch.float32)
    torch.manual_seed(0); tcn = TCN(); opt = torch.optim.Adam(tcn.parameters(), 1e-3); t = time.time()
    for epoch in range(40):
        for idx in torch.randperm(len(Xn)).split(64):
            loss = pinball_loss(tcn(Xn[idx]), Yn[idx]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): out = tcn(torch.tensor(Xtn, dtype=torch.float32)).numpy() * st[:, None, :] + mt[:, None, :]
    results["global TCN + RevIN (median)"] = mae(out[:, 1], Y_te)
    coverage = float(np.mean((Y_te >= out[:, 0]) & (Y_te <= out[:, 2])))
    print(f"TCN trained in {time.time() - t:.0f}s, {sum(p.numel() for p in tcn.parameters()):,} parameters")
    print(f"80 % interval (P10–P90) covers {coverage:.0%} of the actual 2025 prices")
    for k, v in sorted(results.items(), key=lambda kv: kv[1]): print(f"{k:<28} MAE {v:.2f}")

In [ ]:
if ready("RevIN", "causal conv", "pinball loss"):
    i = len(test["DK1"][0]) - 30                                      # a DK1 day near the end of the test period
    plt.figure(figsize=(10, 3.5)); hrs = np.arange(-48, 24)
    plt.plot(hrs[:48], X_te[i, -48:], "k", label="last 48 h"); plt.plot(np.arange(24), Y_te[i], "k--", label="actual")
    plt.plot(np.arange(24), out[i, 1], "C0", label="TCN median"); plt.fill_between(np.arange(24), out[i, 0], out[i, 2], alpha=.25, label="P10–P90")
    plt.plot(np.arange(24), glob(X_te[i:i + 1])[0], "C1", label="global linear"); plt.axvline(0, c="gray", lw=.5); plt.legend(fontsize=8); plt.title("DK1, one test day"); plt.show()
    check("neural model is competitive", lambda: results["global TCN + RevIN (median)"] < results["seasonal naive (yesterday)"],
          "The TCN should at least beat seasonal naive.")
    print("→ Compare the TCN with the global linear model honestly. And is the 80 % interval really 80 %? (If not: calibrate it, e.g. widen it on validation data.)")

<details><summary>▶ Solution</summary>

```python
def revin(X, eps=1e-3):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + eps
    return (X - mu) / sd, (mu, sd)
```

```python
    def forward(self, x):   # CausalConv1d
        return self.conv(F.pad(x, (self.pad, 0)))
```

```python
def pinball_loss(pred, y, quantiles=(0.1, 0.5, 0.9)):
    loss = 0.0
    for i, q in enumerate(quantiles):
        u = y - pred[:, i]
        loss = loss + torch.mean(torch.maximum(q * u, (q - 1) * u))
    return loss
```
</details>

## Stretch goals
- **Add covariates:** the day-ahead wind/solar forecasts from [[Danish Electricity Prices]] as extra input channels. That's where the real gains are (MAE 29.6 → 21.8 in that project).
- **Area embedding:** give the model a learned vector per price area (a static covariate). Does it help the global model?
- **Calibrate the intervals:** conformal prediction on a validation month, i.e. widen P10–P90 until it covers 80 %.
- **N-BEATS or PatchTST** in place of the TCN, using the libraries listed in the note (`neuralforecast`). Do they beat the linear model?